<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/lstm_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Baseline (New Model)

Trains and evaluates an **LSTM** as a new proposed baseline for the error-recognition task
(`error_recognition`), on **Omnivore** features and the **step** split, then compares it against
the already-reproduced **MLP** and **Transformer** baselines.

This is a separate notebook from the baseline reproduction and error-category-analysis notebooks
and does not modify either of them.

<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/notebooks/lstm_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LSTM Baseline

To provide an additional temporal baseline, we implement a 2-layer LSTM
(hidden size 512) followed by a linear classification head.

The model processes the sequence of Omnivore sub-segment features corresponding
to each procedural step and predicts whether the step was executed correctly or
contains an error.

The existing project data-loading, training, and evaluation utilities are reused
without modifying the original MLP and Transformer implementations.

### Batch size

Training uses `batch_size=1` because the current `collate_fn` concatenates
sub-segment sequences instead of padding them into separate batched sequences.
Using a batch size of 1 ensures that the LSTM hidden state is reset between
independent procedural steps and that each input sequence corresponds to one
actual step.

## 1. Setup (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone --recursive --branch zeynep-september https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git \
/content/code

Cloning into '/content/code'...
remote: Enumerating objects: 911, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 911 (delta 102), reused 88 (delta 87), pack-reused 786 (from 2)
Receiving objects: 100% (911/911), 96.54 MiB | 22.04 MiB/s, done.
Resolving deltas: 100% (492/492), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 3.09 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [3]:
%cd /content/code

/content/code


In [4]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU."
)

DEVICE = "cuda"

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [5]:
!pip install -q torcheval pyrebase4 yacs loguru wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


## 2. Omnivore features

Same layout used in the other notebooks in this repo: `omnivore.zip` already contains an
`omnivore/` folder, so we extract straight into `data/video` -> `data/video/omnivore`, which is
where `CaptainCookStepDataset` expects features
(`segment_features_directory="data/"` + `"video"` + backbone).

In [6]:
import os

DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
FEATURES_ZIP = f"{DRIVE_BASE_PATH}/1s.zip"

CODE_DIR = "/content/code"
TEMP_DIR = "/content/temp_features"
VIDEO_DIR = f"{CODE_DIR}/data/video"

!rm -rf "{TEMP_DIR}"
!mkdir -p "{TEMP_DIR}"
!mkdir -p "{VIDEO_DIR}"

!cp "{FEATURES_ZIP}" /content/1s.zip
!unzip -q -o /content/1s.zip -d "{TEMP_DIR}"
!unzip -q -o "{TEMP_DIR}/1s/video/omnivore.zip" -d "{VIDEO_DIR}"

!rm /content/1s.zip
!rm -rf "{TEMP_DIR}"

files = os.listdir(f"{VIDEO_DIR}/omnivore")
print(f"Omnivore features ready: {len(files)} files")

Omnivore features ready: 384 files


## 3. LSTM model

We introduce an LSTM-based temporal baseline for supervised error recognition.

The model consists of a 2-layer `nn.LSTM` with hidden size 512 operating on the
1024-dimensional Omnivore feature sequence corresponding to each procedural step.
A linear classification head produces one logit per sub-segment, following the
same per-subsegment supervision setup used by the existing baselines.

The LSTM is instantiated directly in this notebook, while the existing repository
utilities for data loading, feature dimensions, training, and evaluation are reused.

In [13]:
sys.path.insert(0, ".")

from types import SimpleNamespace
from constants import Constants as const
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn
from core.models.blocks import fetch_input_dim
from base import train_epoch, test_er_model
from torch.utils.data import DataLoader
import torch.nn as nn


class LSTM(nn.Module):

    def __init__(self, config):
        super(LSTM, self).__init__()
        self.config = config
        input_dim = fetch_input_dim(config)
        hidden_dim = 512
        num_layers = 2

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.5 if num_layers > 1 else 0,
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_dim) or (seq_len, input_dim)
        x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)

        # 2D input (seq_len, input_dim) -> treat as a single sequence, batch size 1
        is_unbatched = x.dim() == 2
        if is_unbatched:
            x = x.unsqueeze(0)

        out, (h_n, c_n) = self.lstm(x)   # out: (batch, seq_len, hidden_dim)
        logits = self.fc(out)            # (batch, seq_len, 1) -- one prediction per time step

        if is_unbatched:
            logits = logits.squeeze(0)   # (seq_len, 1)

        return logits

## 4. Train + evaluate (val selects, test reported once)

Same train/val/test workflow already used for the error-category notebook, no test leakage:

1. Train on the `train` split (`batch_size=1`, see note above).
2. After every epoch, evaluate on `val` and record val AUC.
3. Keep the model weights from the epoch with the best val AUC (in memory, not written to disk).
4. Restore those best-on-validation weights after training finishes.
5. Evaluate that restored model **once** on `test` — the only time test data is touched.
6. Return the final test metrics, plus the selected epoch and its validation AUC.

In [12]:
BACKBONE = const.OMNIVORE
SPLIT = const.STEP_SPLIT
TASK_NAME = const.ERROR_RECOGNITION
NUM_EPOCHS = 15     # sufficient training horizon while selecting the best epoch on validation AUC
BATCH_SIZE = 1      # not a tunable hyperparameter here -- see note in Section 3 setup
THRESHOLD = 0.6     # consistent with step-split thresholds used elsewhere in this repo


def run_lstm_experiment(num_epochs=NUM_EPOCHS, threshold=THRESHOLD):
    config = SimpleNamespace(
        backbone=BACKBONE,
        modality="video",
        segment_features_directory="data/",
        split=SPLIT,
        task_name=TASK_NAME,
        error_category=None,
        seed=1000,
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    torch.manual_seed(config.seed)

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    model = LSTM(config).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.5], device=DEVICE))

    best_val_auc = -1.0
    best_epoch = None
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):
        train_epoch(model, DEVICE, train_loader, optimizer, epoch, criterion)

        # Model selection uses VAL only. TEST is not touched here.
        _, _, val_metrics = test_er_model(
            model, val_loader, criterion, DEVICE, phase="val", threshold=threshold
        )
        val_auc = float(val_metrics["auc"])
        print(f"  epoch {epoch}: val AUC = {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}

    # Restore the best-on-validation weights, then evaluate on TEST exactly once.
    model.load_state_dict(best_state_dict)
    _, _, test_metrics = test_er_model(
        model, test_loader, criterion, DEVICE, phase="test", threshold=threshold
    )

    print(f"  selected best epoch = {best_epoch} (val AUC = {best_val_auc:.4f})")
    print(f"  final TEST metrics: {test_metrics}")

    return {
        "best_epoch": best_epoch,
        "val_auc": best_val_auc,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "auc": test_metrics["auc"],
    }

### Smoke test

Run a couple of quick epochs first to confirm the val-selection / single-test-eval workflow
before the full run. Check the output for:
- a `val AUC = ...` line per epoch,
- one `selected best epoch` line,
- exactly one `final TEST metrics` line.

In [9]:
_smoke_metrics = run_lstm_experiment(num_epochs=3)
print("\nReturned dict:")
print(_smoke_metrics)

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 3751/3752, Loss: 0.018595: 100%|██████████| 3752/3752 [01:07<00:00, 55.64it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 99.99it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.5133639455313286), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.3076923076923077, 'recall': 0.016260162601626018, 'f1': 0.03088803088803089, 'accuracy': 0.6757105943152455, 'auc': np.float64(0.47557896033505787), 'pr_auc': tensor(0.3177)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.4756


Train Epoch: 2, Progress: 3751/3752, Loss: 0.142621: 100%|██████████| 3752/3752 [01:03<00:00, 59.32it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 103.58it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.5155273808792523), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.48, 'recall': 0.04878048780487805, 'f1': 0.08856088560885608, 'accuracy': 0.6808785529715762, 'auc': np.float64(0.5130112096575511), 'pr_auc': tensor(0.3257)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5130


Train Epoch: 3, Progress: 3751/3752, Loss: 0.199595: 100%|██████████| 3752/3752 [01:04<00:00, 58.16it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 105.99it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.5057620985883047), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.3125, 'recall': 0.02032520325203252, 'f1': 0.03816793893129771, 'accuracy': 0.6744186046511628, 'auc': np.float64(0.5128726287262872), 'pr_auc': tensor(0.3177)}
----------------------------------------------------------------
  epoch 3: val AUC = 0.5129


test Progress: 42347/798: 100%|██████████| 798/798 [00:08<00:00, 91.18it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.7200746215788604, 'auc': np.float64(0.5260410094147627), 'pr_auc': tensor(0.2799)}
test Step Level Metrics: {'precision': 0.391304347826087, 'recall': 0.03614457831325301, 'f1': 0.0661764705882353, 'accuracy': 0.681704260651629, 'auc': np.float64(0.5590522380962831), 'pr_auc': tensor(0.3149)}
----------------------------------------------------------------
  selected best epoch = 2 (val AUC = 0.5130)
  final TEST metrics: {'precision': 0.391304347826087, 'recall': 0.03614457831325301, 'f1': 0.0661764705882353, 'accuracy': 0.681704260651629, 'auc': np.float64(0.5590522380962831), 'pr_auc': tensor(0.3149)}

Returned dict:
{'best_epoch': 2, 'val_auc': 0.5130112096575511, 'accuracy': 0.681704260651629, 'precision': 0.391304347826087, 'recall': 0.03614457831325301, 'f1': 0.0661764705882353, 'auc': np.float64(0.5590522380962831)}


### Full run

In [10]:
lstm_metrics = run_lstm_experiment()
lstm_metrics

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 3751/3752, Loss: 0.018595: 100%|██████████| 3752/3752 [01:02<00:00, 59.83it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 101.59it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.5133639455313286), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.3076923076923077, 'recall': 0.016260162601626018, 'f1': 0.03088803088803089, 'accuracy': 0.6757105943152455, 'auc': np.float64(0.47557896033505787), 'pr_auc': tensor(0.3177)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.4756


Train Epoch: 2, Progress: 3751/3752, Loss: 0.142621: 100%|██████████| 3752/3752 [01:02<00:00, 60.25it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 101.66it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.5155273808792523), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.48, 'recall': 0.04878048780487805, 'f1': 0.08856088560885608, 'accuracy': 0.6808785529715762, 'auc': np.float64(0.5130112096575511), 'pr_auc': tensor(0.3257)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5130


...

Train Epoch: 15, Progress: 3751/3752, Loss: 0.067803: 100%|██████████| 3752/3752 [01:02<00:00, 59.84it/s]
val Progress: 34385/774: 100%|██████████| 774/774 [00:07<00:00, 107.53it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.6134651737676312, 'auc': np.float64(0.521894734445941), 'pr_auc': tensor(0.3865)}
val Step Level Metrics: {'precision': 0.4166666666666667, 'recall': 0.02032520325203252, 'f1': 0.03875968992248062, 'accuracy': 0.6795865633074936, 'auc': np.float64(0.48277746982015274), 'pr_auc': tensor(0.3198)}
----------------------------------------------------------------
  epoch 15: val AUC = 0.4828


test Progress: 42347/798: 100%|██████████| 798/798 [00:08<00:00, 91.07it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.7200746215788604, 'auc': np.float64(0.4922489934558411), 'pr_auc': tensor(0.2799)}
test Step Level Metrics: {'precision': 0.3130081300813008, 'recall': 0.927710843373494, 'f1': 0.46808510638297873, 'accuracy': 0.34210526315789475, 'auc': np.float64(0.4967776387883044), 'pr_auc': tensor(0.3129)}
----------------------------------------------------------------
  selected best epoch = 10 (val AUC = 0.5484)
  final TEST metrics: {'precision': 0.3130081300813008, 'recall': 0.927710843373494, 'f1': 0.46808510638297873, 'accuracy': 0.34210526315789475, 'auc': np.float64(0.4967776387883044), 'pr_auc': tensor(0.3129)}


{'best_epoch': 10,
 'val_auc': 0.5484032397142153,
 'accuracy': 0.34210526315789475,
 'precision': 0.3130081300813008,
 'recall': 0.927710843373494,
 'f1': 0.46808510638297873,
 'auc': np.float64(0.4967776387883044)}

## 5. Comparison against the reproduced MLP and Transformer baselines

MLP/Transformer rows are the Omnivore + step-split numbers already reproduced separately (evaluated
from the CaptainCook4D pretrained checkpoints, not retrained in this notebook, per instructions).
Not run here -- hardcoded from that earlier run.

In [11]:
import pandas as pd

REPRODUCED_BASELINES = [
    {"Model": "MLP", "Accuracy": 71.05, "Precision": 66.07, "Recall": 14.86, "F1": 24.26, "AUC": 75.74},
    {"Model": "Transformer", "Accuracy": 69.92, "Precision": 51.56, "Recall": 59.84, "F1": 55.39, "AUC": 75.62},
]

rows = list(REPRODUCED_BASELINES) + [{
    "Model": "LSTM",
    "Accuracy": round(float(lstm_metrics["accuracy"]) * 100, 2),
    "Precision": round(float(lstm_metrics["precision"]) * 100, 2),
    "Recall": round(float(lstm_metrics["recall"]) * 100, 2),
    "F1": round(float(lstm_metrics["f1"]) * 100, 2),
    "AUC": round(float(lstm_metrics["auc"]) * 100, 2),
}]

comparison_df = pd.DataFrame(rows)[["Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]]
comparison_df

,Model,Accuracy,Precision,Recall,F1,AUC
0,MLP,71.05,66.07,14.86,24.26,75.74
1,Transformer,69.92,51.56,59.84,55.39,75.62
2,LSTM,34.21,31.30,92.77,46.81,49.68
